In [7]:
# dependencies & packages
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split, StratifiedKFold

In [8]:
# read data
client_test = pd.read_csv("Data/client_test.csv", low_memory=False)
client_train = pd.read_csv("Data/client_train.csv", low_memory=False)
invoice_test = pd.read_csv("Data/invoice_test.csv", low_memory=False)
invoice_train = pd.read_csv("Data/invoice_train.csv", low_memory=False)

In [9]:
# inspect data
def look(name, df):
    print("=" * 80)
    print(name, df.shape)
    display(df.head(3))
    display(pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "n_missing": df.isna().sum(),
        "pct_missing": (df.isna().mean() * 100).round(2),
        "n_unique": df.nunique(),
    }))
    print("duplicate rows:", df.duplicated().sum())

look("client_test", client_test)
look("client_train", client_train)
look("invoice_test", invoice_test)
look("invoice_train", invoice_train)

client_test (58069, 5)


,disrict,client_id,client_catg,region,creation_date
0,62,test_Client_0,11,307,28/05/2002
1,69,test_Client_1,11,103,06/08/2009
2,62,test_Client_10,11,310,07/04/2004


,dtype,n_missing,pct_missing,n_unique
disrict,int64,0,0.0,4
client_id,object,0,0.0,58069
client_catg,int64,0,0.0,3
region,int64,0,0.0,24
creation_date,object,0,0.0,7388


duplicate rows: 0
client_train (135493, 6)


,disrict,client_id,client_catg,region,creation_date,target
0,60,train_Client_0,11,101,31/12/1994,0.0
1,69,train_Client_1,11,107,29/05/2002,0.0
2,62,train_Client_10,11,301,13/03/1986,0.0


,dtype,n_missing,pct_missing,n_unique
disrict,int64,0,0.0,4
client_id,object,0,0.0,135493
client_catg,int64,0,0.0,3
region,int64,0,0.0,25
creation_date,object,0,0.0,8088
target,float64,0,0.0,2


... (3 outputs left out)

duplicate rows: 8
invoice_train (4476749, 16)


,client_id,invoice_date,tarif_type,counter_number,counter_statue,counter_code,reading_remarque,counter_coefficient,consommation_level_1,consommation_level_2,consommation_level_3,consommation_level_4,old_index,new_index,months_number,counter_type
0,train_Client_0,2014-03-24,11,1335667,0,203,8,1,82,0,0,0,14302,14384,4,ELEC
1,train_Client_0,2013-03-29,11,1335667,0,203,6,1,1200,184,0,0,12294,13678,4,ELEC
2,train_Client_0,2015-03-23,11,1335667,0,203,8,1,123,0,0,0,14624,14747,4,ELEC


,dtype,n_missing,pct_missing,n_unique
client_id,object,0,0.0,135493
invoice_date,object,0,0.0,8275
tarif_type,int64,0,0.0,17
counter_number,int64,0,0.0,201893
counter_statue,object,0,0.0,12
counter_code,int64,0,0.0,42
reading_remarque,int64,0,0.0,8
counter_coefficient,int64,0,0.0,16
consommation_level_1,int64,0,0.0,8295
consommation_level_2,int64,0,0.0,12576


duplicate rows: 11


In [ ]:
# The 8 and 11 duplicate invoice rows in invoice are negligible. 
# --> leaving them in for now. to revisit when we build features

In [10]:
# target balance
print(client_train["target"].value_counts())
print(client_train["target"].value_counts(normalize=True).round(4))

# uniqueness: client files should have one row per client
print("\nclient_train ids unique:", client_train["client_id"].is_unique)
print("client_test ids unique: ", client_test["client_id"].is_unique)

# overlaps
tr, te = set(client_train["client_id"]), set(client_test["client_id"])
itr, ite = set(invoice_train["client_id"]), set(invoice_test["client_id"])
print("\ntrain & test clients overlap:  ", len(tr & te))
print("train clients w/o invoices:    ", len(tr - itr))
print("test clients w/o invoices:     ", len(te - ite))
print("invoice_train ids not in train:", len(itr - tr))
print("invoice_test ids not in test:  ", len(ite - te))
print("invoices per client (median), train / test:",
      invoice_train.groupby("client_id").size().median(),
      invoice_test.groupby("client_id").size().median())

# date ranges
for name, df in [("client_train", client_train), ("client_test", client_test),
                 ("invoice_train", invoice_train), ("invoice_test", invoice_test)]:
    for c in df.columns:
        if "date" in c.lower():
            d = pd.to_datetime(df[c], errors="coerce", dayfirst=True, format="mixed")
            print(f"{name}.{c}: {d.min().date()} -> {d.max().date()}  (unparseable: {d.isna().sum()})")

target
0.0    127927
1.0      7566
Name: count, dtype: int64
target
0.0    0.9442
1.0    0.0558
Name: proportion, dtype: float64

client_train ids unique: True
client_test ids unique:  True

train & test clients overlap:   0
train clients w/o invoices:     0
test clients w/o invoices:      0
invoice_train ids not in train: 0
invoice_test ids not in test:   0
invoices per client (median), train / test: 30.0 30.0
client_train.creation_date: 1977-02-05 -> 2019-09-10  (unparseable: 0)
client_test.creation_date: 1977-02-05 -> 2019-08-26  (unparseable: 0)
invoice_train.invoice_date: 1977-06-09 -> 2019-12-07  (unparseable: 0)
invoice_test.invoice_date: 1977-07-13 -> 2019-12-07  (unparseable: 0)


In [11]:
# holdout + 5 committed folds

SEED, N_FOLDS, HOLDOUT_FRAC = 42, 5, 0.20

def make_splits(client_df, seed=SEED, n_folds=N_FOLDS, holdout_frac=HOLDOUT_FRAC):
    """One row per client: split in {dev, holdout}; fold in 0..n_folds-1 (-1 for holdout)."""
    client_df = client_df.reset_index(drop=True)
    y = client_df["target"].astype(int)

    dev_idx, hold_idx = train_test_split(
        client_df.index, test_size=holdout_frac, stratify=y, random_state=seed
    )
    splits = pd.DataFrame({"client_id": client_df["client_id"], "split": "dev", "fold": -1})
    splits.loc[hold_idx, "split"] = "holdout"

    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=seed)
    for k, (_, val) in enumerate(skf.split(dev_idx, y.loc[dev_idx])):
        splits.loc[dev_idx[val], "fold"] = k
    return splits

splits = make_splits(client_train)

# sanity checks
assert splits["client_id"].is_unique
assert set(splits["client_id"]) == set(client_train["client_id"])
assert (splits.loc[splits.split == "dev", "fold"] >= 0).all()

chk = splits.merge(client_train[["client_id", "target"]], on="client_id")
print(chk.groupby(["split", "fold"]).agg(n=("target", "size"), fraud_rate=("target", "mean")).round(4))

Path("splits").mkdir(exist_ok=True)
if not Path("splits/splits.csv").exists():
    splits.to_csv("splits/splits.csv", index=False)
else:
    print("splits.csv already exists, not overwriting")

                  n  fraud_rate
split   fold                   
dev      0    21679      0.0558
         1    21679      0.0559
         2    21679      0.0559
         3    21679      0.0559
         4    21678      0.0558
holdout -1    27099      0.0558


In [12]:
# stubs
def fake_like(df, n, id_col="client_id", ids=None, seed=0):
    rng = np.random.default_rng(seed)
    out = {}
    for c in df.columns:
        s = df[c].dropna()
        if c == id_col and ids is not None:
            out[c] = ids
        elif "date" in c.lower():
            d = pd.to_datetime(s, errors="coerce", dayfirst=True, format="mixed").dropna()
            out[c] = pd.to_datetime(rng.integers(d.min().value, d.max().value, n)).strftime("%Y-%m-%d")
        elif s.nunique() <= 30:
            out[c] = rng.choice(s.unique(), n)
        elif pd.api.types.is_numeric_dtype(s):
            vals = rng.normal(0, 1, n) * (s.std() or 1) + s.mean()
            out[c] = vals.round().astype(s.dtype) if pd.api.types.is_integer_dtype(s) else vals
        else:
            out[c] = [f"{c}_{i}" for i in rng.integers(0, n, n)]
    return pd.DataFrame(out)

def fake_invoices(real_invoice, client_ids, per_client=30, seed=0):
    ids = np.repeat(client_ids, per_client)
    return fake_like(real_invoice, len(ids), ids=ids, seed=seed)

rng = np.random.default_rng(0)
N_TR, N_TE = 600, 250
tr_ids = np.array([f"stub_train_{i}" for i in range(N_TR)])
te_ids = np.array([f"stub_test_{i}" for i in range(N_TE)])

stub_client_train = fake_like(client_train, N_TR, ids=tr_ids, seed=1)
stub_client_train["target"] = (rng.random(N_TR) < 0.06).astype(int)
stub_client_test = fake_like(client_test, N_TE, ids=te_ids, seed=2)

# every client has 30 invoices, matching the real data
stub_invoice_train = fake_invoices(invoice_train, tr_ids, seed=3)
stub_invoice_test = fake_invoices(invoice_test, te_ids, seed=4)

# splits stub built by the same function as the real one
stub_splits = make_splits(stub_client_train)

Path("stubs").mkdir(exist_ok=True)
for name, df in [("client_train", stub_client_train), ("client_test", stub_client_test),
                 ("invoice_train", stub_invoice_train), ("invoice_test", stub_invoice_test),
                 ("splits", stub_splits)]:
    df.to_csv(f"stubs/{name}.csv", index=False)

# schema check against the real files
for name, real, fake in [("client_train", client_train, stub_client_train),
                         ("client_test", client_test, stub_client_test),
                         ("invoice_train", invoice_train, stub_invoice_train),
                         ("invoice_test", invoice_test, stub_invoice_test)]:
    assert list(real.columns) == list(fake.columns), name
print("stubs OK")

stubs OK
